Testing to ensre that the implimentation of the Pauli Z Hamiltoinan is correct: 

In [1]:
import numpy as np
import pandas as pd
import networkx as nx
from itertools import product

Add funcitons for graph creation: 

In [2]:
def add_edges(graph: nx.Graph, edges: list, weights: list):
    """Add edges with weights to the graph."""
    for edge, weight in zip(edges, weights):
        graph.add_edge(edge[0], edge[1], weight=weight)

def construct_graph_from_csv(filename: str) -> nx.Graph:
    """
    Construct a NetworkX graph from a CSV file with columns Node1,Node2,Weight.
    Node1 and Node2 has to be ints, while Weight can be any numeric type.

    LLM-assisted
    ------------
    Tool: Github Copilot (2026)
    Created print statement
    """
    network_graph = nx.Graph()
    graph = pd.read_csv(filename)
    edges   = [(row['Node1'], row['Node2']) for _, row in graph.iterrows()]
    weights = [row['Weight'] for _, row in graph.iterrows()]

    add_edges(network_graph, edges, weights)

    print(f"Graph has {network_graph.number_of_nodes()} nodes and {network_graph.number_of_edges()} edges.")

    nodelist = sorted(network_graph.nodes())
    return np.array(nx.adjacency_matrix(network_graph, nodelist=nodelist, weight='weight').todense())

Add modularity Hamiltoians to be tested:     

In [3]:
def ising_hamiltonian_k2_modularity(A: np.ndarray, alpha: float) -> tuple:
    """
    Derive the coupling matrix J and constant offset for the k=2 modularity
    Ising Hamiltonian H = sum_{i<j} J_ij s_i s_j + const.

    The modularity matrix B_ij = (A_ij - alpha * k_i k_j / 2m) / 2m is
    split into an interaction part (off-diagonal, i < j) and a constant
    contribution from the diagonal.
    """
    num_nodes = len(A)
    m = np.sum(A) / 2
    k = A.sum(axis=1)
    B = (A - alpha * np.outer(k, k) / (2 * m)) / (2 * m)

    J = np.zeros((num_nodes, num_nodes))
    const = 0.0

    for i in range(num_nodes):
        for j in range(i + 1, num_nodes):
            J[i, j] += B[i, j]
        const += B[i, i]

    return J, const / 2

In [4]:
# Pauli matrices
I2 = np.eye(2, dtype=complex)
Zp = np.array([[1, 0], [0, -1]], dtype=complex)

def kron_two(op_i: np.ndarray, qi: int, op_j: np.ndarray, qj: int, n_qubits: int) -> np.ndarray:
    """Embed two single-qubit operators on different qubits."""
    ops = [I2] * n_qubits
    ops[n_qubits - 1 - qi] = op_i
    ops[n_qubits - 1 - qj] = op_j
    result = ops[0]
    for o in ops[1:]:
        result = np.kron(result, o)
    return result

def pauli_z_hamiltonian(A: np.ndarray, alpha: float) -> np.ndarray:
    """
    Construct the full 2^n x 2^n cost Hamiltonian H_C in the computational basis.

    Classical spin variables s_i in {-1, +1} are promoted to Pauli-Z operators,
    and two-spin interactions J_ij s_i s_j become ZZ tensor products.
    H_C is diagonal in the computational basis — its eigenvalues are the
    modularity values of all 2^n spin configurations.
    """
    num_nodes = len(A)
    DIM = 2 ** num_nodes
    H_C = np.zeros((DIM, DIM), dtype=complex)
    J, const = ising_hamiltonian_k2_modularity(A, alpha)

    # Sum ZZ interaction terms weighted by J_ij
    for i in range(num_nodes):
        for j in range(i + 1, num_nodes):
            ZZ = kron_two(Zp, i, Zp, j, num_nodes)
            H_C += J[i, j] * ZZ

    # Add the constant offset as a scaled identity
    H_C += const * np.eye(DIM, dtype=complex)

    return H_C

Add a base implimentation of the modularity calculation: 

In [5]:
def modularity_calc(A: np.ndarray, alpha: float, x: np.ndarray) -> float:
    k = A.sum(axis=1)
    deg_sum = k.sum()
    m = deg_sum / 2
    norm = 1 / deg_sum**2

    Q = 0.0
    for comm_id in np.unique(x):
        members = np.where(x == comm_id)[0]
        # L_c: intra-community edge weight (each edge once, like nx.G.edges())
        subA = A[np.ix_(members, members)]
        L_c = subA.sum() / 2  # divide by 2 because A is symmetric
        k_c = k[members].sum()
        Q += L_c / m - alpha * k_c**2 * norm

    return Q

Create a function to find the energy of the Pauli Z Hamiltoian from a bitstring: 

In [6]:
def bitstring_energy(H_C: np.ndarray, x: np.array, n_qubits: int) -> float:
    """
    Get energy of a computational basis state from the Pauli-Z Hamiltonian.
    x: binary array of length n, e.g. [0, 1, 0, 1]
    """
    x = np.flip(x)
    # Map x to a bitstring: 
    x = ''.join(x.astype(int).astype(str))
    idx = 0
    while True: 
        format(idx, f'0{n_qubits}b')
        if format(idx, f'0{n_qubits}b') == x:
            break
        idx += 1

    return H_C[idx, idx].real

Create a function for testing given a graph. Use the networkx's modularity calculation as a baseline for comparison: 

In [ ]:
def test_hamiltonian_construction(graph_path: str):
    A = construct_graph_from_csv(graph_path)
    network_graph = nx.from_numpy_array(A)
    N_QUBITS = A.shape[0]

    H, const = ising_hamiltonian_k2_modularity(A, alpha=1.0)
    H_C = pauli_z_hamiltonian(A, alpha=1.0)

    print("Brute-force optimal k-cluster modularity Q calculation: \n")
    print("-----------------------------\n")

    prods = np.array(list(product(range(2), repeat=N_QUBITS)))

    sols = []
    for i, x in enumerate(prods):
        z_full = np.array([2*x_i - 1 for x_i in x])

        obj_val = modularity_calc(A, 1.0, x=x)
        networkx_modularity = nx.algorithms.community.quality.modularity(network_graph, [np.where(x == 0)[0].tolist(), np.where(x == 1)[0].tolist()])
        H_modularity = z_full @ H @ z_full + const
        
        # Pauli Z Hamiltonian modularity
        H_C_modularity = bitstring_energy(H_C, x, N_QUBITS)

        # Check if all the modularity calculations match
        sols.append(obj_val)
        assert np.isclose(obj_val, networkx_modularity), f"Modularity mismatch for state {x}"
        assert np.isclose(obj_val, H_modularity), f"Hamiltonian modularity mismatch for state {x}"
        assert np.isclose(obj_val, H_C_modularity), f"Pauli Z Hamiltonian modularity mismatch for state {x}"

    print("All modularity calculations match for all configurations!")

In [10]:
test_hamiltonian_construction("graphs/XXSgraph.csv")
test_hamiltonian_construction("graphs/SmallKarateClub.csv")
test_hamiltonian_construction("graphs/HomeMadeGraph1.csv")
test_hamiltonian_construction("graphs/HomeMadeGraph2.csv")

Graph has 5 nodes and 5 edges.
[-0.   -0.02 -0.08  0.22 -0.18 -0.32 -0.1   0.08 -0.08 -0.18 -0.32 -0.1
 -0.1  -0.32 -0.18 -0.08 -0.08 -0.18 -0.32 -0.1  -0.1  -0.32 -0.18 -0.08
  0.08 -0.1  -0.32 -0.18  0.22 -0.08 -0.02 -0.  ]
Brute-force optimal k-cluster modularity Q calculation: 

-----------------------------

All modularity calculations match for all configurations!
Graph has 12 nodes and 22 edges.
[ 0.     -0.1033 -0.0165 ... -0.0165 -0.1033  0.    ]
Brute-force optimal k-cluster modularity Q calculation: 

-----------------------------

All modularity calculations match for all configurations!
Graph has 10 nodes and 15 edges.
[-0.     -0.02   -0.0089 ... -0.0089 -0.02   -0.    ]
Brute-force optimal k-cluster modularity Q calculation: 

-----------------------------

All modularity calculations match for all configurations!
Graph has 12 nodes and 14 edges.
[-0.     -0.0102 -0.0026 ... -0.0026 -0.0102 -0.    ]
Brute-force optimal k-cluster modularity Q calculation: 

--------------